In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor
from scipy.stats import norm
from scipy.stats import binomtest

#sns.set_style("white")
#sns.set(font_scale = 1.5)
#sns.set_style("white")

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

#For ML, seems like SpecSup250, Per90, and SpecSup250 + Per90 are the way to go
#If we decide to do the whole controlling for effect size when assigning to genes/species for NC PhyloP, we will need to alter code to do it for just species for ML

gobp = pd.read_csv("GOBP_AccelEvol_Input_FiltForAccelEvol.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("HPO_AccelEvol_Input_FiltForAccelEvol.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")
d_abrev = {"LiangSteinNeuron":"FC exc. neur.", "FetalChondrocytes":"F chond.", "SertoliMale":"FG sertoli", "preGC_IIaFemale":"FG preGC IIa",\
      "NeuralFemale":"FG neur.", "FetalGonadImmuneFemale":"FG immune", "VIP":"AC VIP inh. neur.", "LiangSteinProgenitor":"FC prog.",\
      "AdultHeartVentricularCardiomyocyte":"AH cardiomyo.", "AdultLoopOfHenle":"AK loop of henle", "FetalBrainNeurGlioblast_CB_VZ":"FCB glioblast",\
     "AdultProximalTubule":"AK prox. tub.", "FetalLeydigMale":"FG leydig", "SST":"AC SST inh neur.", "KosoyRoussosControlMicroglia":"AC microglia",\
     "FetalBrainFloorPlate":"FB fl. plate", "FetalArterialECs":"FH endoth.", "ASCT":"AC astro.", "FetalBrainCOP":"FB COP",\
     "AMY":"AA neur.", "PVALB":"AC PVALB inh neur.", "ITL23":"AC L2-3 IT neur.", "FetalBrainNeurCB_GNP_IPC_1":"FB inter. prog.", "FetalBrainNeurDAergic":"FB DA neur.",\
      "OGC":"AC Oligo.", "D1Pu":"AP D1 inh neur.", "FetalBrainNeurSerotonergic":"FB 5-HT neur.", "FetalBrainNeurDRG_2":"FS DRG neur.",\
      "FetalHeartPericytes":"FH peri.", "FetalHeartEndocardium":"FH endocard.", "FetalHeartCardiacFibroblasts":"FH fibro.", "FetalBrainNeurPurkinje_6":"FCB Purk. inh neur.",\
      "AdultHeartSmoothMuscle":"AH smooth musc.", "FetalBrainRoofPlate":"FB ro. plate"}


In [ ]:
z = pd.read_csv("LiangSteinNeuron_AllSitesToDownload.txt.gz", sep = "\t")
z["AncNuc"] = [x[1] for x in z["AncTrinuc"]]
z["DerNuc"] = [x[1] for x in z["DerTrinuc"]]


In [ ]:
#Checking that this is not the result of ChromBPNet generally predictin weak to strong increasing accessibility and strong to weak decreasing accessibility
#Result generally gets stronger at 0.5 rather than 0.25

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 6) & (z2["PhyloP447"] < 12)]
z2 = z2[((z2["AncNuc"] == "G") | (z2["AncNuc"] == "C")) & ((z2["DerNuc"] == "A") | (z2["DerNuc"] == "T"))]
z2 = z2[z2["Derived"] == "H"]

up_cons = z2[z2["logfc"] < -0.5].shape[0]
down_cons = z2[z2["logfc"] > 0.5].shape[0]

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 0) & (z2["PhyloP447"] < 1)]
z2 = z2[((z2["AncNuc"] == "G") | (z2["AncNuc"] == "C")) & ((z2["DerNuc"] == "A") | (z2["DerNuc"] == "T"))]
z2 = z2[z2["Derived"] == "H"]

up_ncons = z2[z2["logfc"] < -0.25].shape[0]
down_ncons = z2[z2["logfc"] > 0.25].shape[0]

print("GC -> AT", fisher_exact([[down_cons, up_cons], [down_ncons, up_ncons]]), [[down_cons, up_cons], [down_ncons, up_ncons]])

In [ ]:
#Checking that this is not the result of ChromBPNet generally predicting weak to strong increasing accessibility and strong to weak decreasing accessibility
#Result generally gets stronger at 0.5 rather than 0.25
z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 6) & (z2["PhyloP447"] < 12)]
z2 = z2[((z2["AncNuc"] == "G") | (z2["AncNuc"] == "C")) & ((z2["DerNuc"] == "G") | (z2["DerNuc"] == "C"))]
z2 = z2[z2["Derived"] == "H"]

up_cons = z2[z2["logfc"] < -0.5].shape[0]
down_cons = z2[z2["logfc"] > 0.5].shape[0]

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 0) & (z2["PhyloP447"] < 1)]
z2 = z2[((z2["AncNuc"] == "G") | (z2["AncNuc"] == "C")) & ((z2["DerNuc"] == "G") | (z2["DerNuc"] == "C"))]
z2 = z2[z2["Derived"] == "H"]

up_ncons = z2[z2["logfc"] < -0.25].shape[0]
down_ncons = z2[z2["logfc"] > 0.25].shape[0]

print("GC -> GC", fisher_exact([[down_cons, up_cons], [down_ncons, up_ncons]]), [[down_cons, up_cons], [down_ncons, up_ncons]])

In [ ]:
#Checking that this is not the result of ChromBPNet generally predicting weak to strong increasing accessibility and strong to weak decreasing accessibility
#Result becomes significant, but does get stronger for this too
z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 6) & (z2["PhyloP447"] < 12)]
z2 = z2[((z2["AncNuc"] == "A") | (z2["AncNuc"] == "T")) & ((z2["DerNuc"] == "G") | (z2["DerNuc"] == "C"))]
z2 = z2[z2["Derived"] == "H"]

up_cons = z2[z2["logfc"] < -0.5].shape[0]
down_cons = z2[z2["logfc"] > 0.5].shape[0]

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 0) & (z2["PhyloP447"] < 1)]
z2 = z2[((z2["AncNuc"] == "A") | (z2["AncNuc"] == "T")) & ((z2["DerNuc"] == "G") | (z2["DerNuc"] == "C"))]
z2 = z2[z2["Derived"] == "H"]

up_ncons = z2[z2["logfc"] < -0.25].shape[0]
down_ncons = z2[z2["logfc"] > 0.25].shape[0]

print("AT -> GC", fisher_exact([[down_cons, up_cons], [down_ncons, up_ncons]]), [[down_cons, up_cons], [down_ncons, up_ncons]])

In [ ]:
#Checking that this is not the result of ChromBPNet generally predicting weak to strong increasing accessibility and strong to weak decreasing accessibility
#Stronger at 0.5, but so few subs that it is hard to evaluate
z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 6) & (z2["PhyloP447"] < 12)]
z2 = z2[((z2["AncNuc"] == "A") | (z2["AncNuc"] == "T")) & ((z2["DerNuc"] == "A") | (z2["DerNuc"] == "T"))]
z2 = z2[z2["Derived"] == "H"]

up_cons = z2[z2["logfc"] < -0.25].shape[0]
down_cons = z2[z2["logfc"] > 0.25].shape[0]

z2 = z[z["SpecSup447"] > 250]
z2 = z2[(z2["PhyloP447"] > 0) & (z2["PhyloP447"] < 1)]
z2 = z2[((z2["AncNuc"] == "A") | (z2["AncNuc"] == "T")) & ((z2["DerNuc"] == "A") | (z2["DerNuc"] == "T"))]
z2 = z2[z2["Derived"] == "H"]

up_ncons = z2[z2["logfc"] < -0.25].shape[0]
down_ncons = z2[z2["logfc"] > 0.25].shape[0]

print("AT -> AT", fisher_exact([[down_cons, up_cons], [down_ncons, up_ncons]]), [[down_cons, up_cons], [down_ncons, up_ncons]])

In [ ]:
x = pd.read_csv("HumChp_AccelEvolInput.Ready.JanetSongpIN.bed", sep = "\t", header = None)


In [ ]:
x = x[x[5] != "."]
x = x[x[6] != "."]
x = x[x[6].astype(float) > 250]

In [ ]:
#x = x[x[4] == "C"]
ttest_ind(x[x[5].astype(float) > 5][36], x[x[5].astype(float) < 1][36])

In [ ]:
x = x[(x[8] == "NC")]
x.index = x[0] + ":" + x[2].astype(str)
x = x[[36, 37]]


In [ ]:
x = x.join(z.set_index("Position.1")).dropna(subset = "PhyloP447")

In [ ]:
xc = x[x["Derived"] == "C"]
xh = x[x["Derived"] == "H"]

xh["logfc"] = -xh["logfc"]
x = pd.concat([xh, xc])

In [ ]:
x2 = x[np.abs(x["logfc"]) > 0.5]
pearsonr(x2["logfc"], x2[36])

In [ ]:
sns.regplot(x = list(x2["logfc"]), y = list(x2[36]))

In [ ]:
uu = x2[(x2["logfc"] > 0) & (x2[36] > 0)].shape[0]
ud = x2[(x2["logfc"] > 0) & (x2[36] < 0)].shape[0]
du = x2[(x2["logfc"] < 0) & (x2[36] > 0)].shape[0]
dd = x2[(x2["logfc"] < 0) & (x2[36] < 0)].shape[0]
fisher_exact([[uu, ud], [du, dd]])

In [ ]:
xh = x2[x2["Derived"] == "H"]
xc = x2[x2["Derived"] == "C"]
xcc = xc[xc["PhyloP447"] > 3]
xcnc = xc[xc["PhyloP447"] < 1]
xhc = xh[xh["PhyloP447"] > 3]
xhnc = xh[xh["PhyloP447"] < 1]

up_cons_chimp = xcc[xcc[36] < 0].shape[0]
down_cons_chimp = xcc[xcc[36] > 0].shape[0]
up_ncons_chimp = xcnc[xcnc[36] < 0].shape[0]
down_ncons_chimp = xcnc[xcnc[36] > 0].shape[0]

up_cons_human = xhc[xhc[36] > 0].shape[0]
down_cons_human = xhc[xhc[36] < 0].shape[0]
up_ncons_human = xhnc[xhnc[36] > 0].shape[0]
down_ncons_human = xhnc[xhnc[36] < 0].shape[0]

In [ ]:
fisher_exact([[down_cons_chimp, up_cons_chimp], [down_ncons_chimp, up_ncons_chimp]])

In [ ]:
fisher_exact([[down_cons_human, up_cons_human], [down_ncons_human, up_ncons_human]])

In [ ]:
up_cons_chimp = xcc[(xcc[36] < 0) & (xcc["logfc"] < 0)].shape[0]
down_cons_chimp = xcc[(xcc[36] > 0) & (xcc["logfc"] > 0)].shape[0]
up_ncons_chimp = xcnc[(xcnc[36] < 0) & (xcnc["logfc"] < 0)].shape[0]
down_ncons_chimp = xcnc[(xcnc[36] > 0) & (xcnc["logfc"] > 0)].shape[0]

up_cons_human = xhc[(xhc[36] > 0) & (xhc["logfc"] > 0)].shape[0]
down_cons_human = xhc[(xhc[36] < 0) & (xhc["logfc"] < 0)].shape[0]
up_ncons_human = xhnc[(xhnc[36] > 0) & (xhnc["logfc"] > 0)].shape[0]
down_ncons_human = xhnc[(xhnc[36] < 0) & (xhnc["logfc"] < 0)].shape[0]

In [ ]:
fisher_exact([[down_cons_chimp, up_cons_chimp], [down_ncons_chimp, up_ncons_chimp]])

In [ ]:
fisher_exact([[down_cons_human, up_cons_human], [down_ncons_human, up_ncons_human]])

In [ ]:
sns.set_style("white")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 4), dpi=600)
dfpp = pd.DataFrame([["Lower CA from\nhuman allele", down_cons_human], ["Higher CA from\nhuman allele", up_cons_human]])
dfpp.columns = ["Effect on CA", "Number of sites"]
sns.barplot(data = dfpp, x = "Effect on CA", y = "Number of sites", hue = "Effect on CA", alpha = 0.85, palette = {"Lower CA from\nhuman allele":"magenta", "Higher CA from\nhuman allele":"green"})
plt.axhline(down_ncons_human/(down_ncons_human + up_ncons_human)*(down_cons_human + up_cons_human), linestyle = "--", color = "black", xmin = 0.055, xmax = 0.4535)
plt.axhline(up_ncons_human/(down_ncons_human + up_ncons_human)*(down_cons_human + up_cons_human), linestyle = "--", color = "black", xmin = 0.5 + 0.055, xmax = 0.5 + 0.4535)
plt.ylabel("Number of sites", size = 11)
plt.xlabel("", size = 11)
plt.xticks(size = 10)
plt.yticks(size = 10)
x1, x22 = 0, 1                 # bar positions
y, h = 83, 0 
#y, h = 21, 0 # height of line + vertical offset
plt.plot([x1, x1, x22, x22],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x22) * 0.5, y - 2,
         "**",
         ha='center', va='bottom', fontsize=14)
plt.title("Experimental validation\nof effects on CA", size = 12)
plt.ylim(0, 90)
#plt.title("Purkinje cells, ATAC peaks near Tenm3", size = 15)

In [ ]:
r = pd.read_csv("../PosSelect_ForPub/piN_RNAseq_JanetSong.csv")
rc = r[r["geneName"].isin(xhc[(xhc[36] < 0) & (xhc["logfc"] < 0)]["NearestGene"])]
rc = rc[rc["sig.ASE"] == True]
rc[rc["log2FoldChange.ASE"] < 0]

In [ ]:
pearsonr(rc["log2FoldChange.ASE"], rc[

In [ ]:
rnc = r[r["sig.ASE"] == True]
rnc[rnc["log2FoldChange.ASE"] > 0].shape

In [ ]:
binomtest(24, 36, p = 4900/(4900 + 5776))

In [ ]:
fig, ax = plt.subplots(figsize=(3, 4), dpi=600)
dfpp = pd.DataFrame([["Lower expr. from\nhuman allele", 24], ["Higher expr. from\nhuman allele", 12]])
dfpp.columns = ["Effect on CA", "Number of sites"]
sns.barplot(data = dfpp, x = "Effect on CA", y = "Number of sites", hue = "Effect on CA", alpha = 0.85, palette = {"Lower expr. from\nhuman allele":"#39BEB1", "Higher expr. from\nhuman allele":"#CD99D8"})
plt.axhline(4900/(4900 + 5776)*36, linestyle = "--", color = "black", xmin = 0.055, xmax = 0.4535)
plt.axhline(5776/(4900 + 5776)*36, linestyle = "--", color = "black", xmin = 0.5 + 0.055, xmax = 0.5 + 0.4535)
plt.ylabel("Number of genes", size = 11)
plt.xlabel("", size = 11)
plt.xticks(size = 10)
plt.yticks(size = 10)
x1, x22 = 0, 1                 # bar positions
#y, h = 83, 0 
y, h = 25, 0 # height of line + vertical offset
plt.plot([x1, x1, x22, x22],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x22) * 0.5, y - 1,
         "*",
         ha='center', va='bottom', fontsize=18)
plt.title("Effects on expression", size = 12)
plt.ylim(0, 28)
#plt.title("Purkinje cells, ATAC peaks near Tenm3", size = 15)

In [ ]:
z2 = z[z["SpecSup447"] > 250]
z2

In [ ]:
z2[(z2["Derived"] == "H") & (z2["PhyloP447"] > 6)].shape[0]

In [ ]:
z2[(z2["Derived"] == "C") & (z2["PhyloP447"] > 6)].shape[0]

In [ ]:
binomtest(23, 23 + 15)

In [ ]:
binomtest(274, 274 + 161, p = 161831/(161831 + 148782))

In [ ]:
binomtest(109, 55 + 109, p = 161831/(161831 + 148782))

In [ ]:
v = pd.read_csv("Phylo_Stuff.txt", sep = "\t")
keep = []
kk = []
for i in np.unique(v["CellType"]):
    vk = v[v["CellType"] == i]
    vk = vk[vk["LFC Left Bin"] == 0.5]
    kk.append([i, np.sum(np.sum(vk[["Human down", "Human up", "Chimp down", "Chimp up"]]))])
    if np.sum(np.sum(vk[["Human down", "Human up", "Chimp down", "Chimp up"]])) >= 10000:
        keep.append(i)
        
v = v[v["CellType"].isin(keep)]
v

In [ ]:
vp = v[((v["PhyloP Left Bin"] == 6) | (v["PhyloP Left Bin"] == 0)) & (v["LFC Left Bin"] == 0.5) & (v["FiltOrNotFilt"] == "NotFilt")]
out = []
for i in np.unique(vp["CellType"]):
    vpc = vp[vp["CellType"] == i]
    vpc6 = vpc[vpc["PhyloP Left Bin"] == 6]
    vpc0 = vpc[vpc["PhyloP Left Bin"] == 0]
    pch = fisher_exact([[vpc6["Human down"].iloc[0], vpc6["Human up"].iloc[0]], [vpc6["Chimp down"].iloc[0], vpc6["Chimp up"].iloc[0]]])
    ph = fisher_exact([[vpc6["Human down"].iloc[0], vpc6["Human up"].iloc[0]], [vpc0["Human down"].iloc[0], vpc0["Human up"].iloc[0]]])
    out.append([i, vpc6["Human down"].iloc[0], vpc6["Human up"].iloc[0], vpc6["Chimp down"].iloc[0], vpc6["Chimp up"].iloc[0], pch[0], pch[1], \
               vpc0["Human down"].iloc[0], vpc0["Human up"].iloc[0], vpc0["Chimp down"].iloc[0], vpc0["Chimp up"].iloc[0], ph[0], ph[1]])
dfp = pd.DataFrame(out)
dfp = dfp.sort_values(6)
dfp.columns = ["Cell type", "6H Down", "6H Up", "6C Down", "6C Up", "OR HC", "p-value HC", "0H Down", "0H Up", "0C Down", "0C Up", "Odds ratio", "p-value"]
dfp["FDR"] = fdrcorrection(dfp["p-value"])[1]
dfp["-Log$_{10}$(FDR)"] = -np.log10(dfp["FDR"])
dfp["Log$_2$(odds ratio)"] = np.log2(dfp["Odds ratio"])
sig = []
for index, row in dfp.iterrows():
    if row["FDR"] < 0.05:
        sig.append("FDR < 0.05")
    else:
        sig.append("Not significant")
dfp["Significance"] = sig
sns.set(font_scale = 1)
sns.set_style("white")
fig, ax = plt.subplots(figsize = (4.2, 4), dpi = 450)
sns.scatterplot(data = dfp, x = "Log$_2$(odds ratio)", y = "-Log$_{10}$(FDR)", color = "magenta")
plt.ylim(0, 29)
plt.title("Subs. in conserved sites\ndecrease CA across cell types")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor
from scipy.stats import norm
from scipy.stats import binomtest

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

#For ML, seems like SpecSup250, Per90, and SpecSup250 + Per90 are the way to go
#If we decide to do the whole controlling for effect size when assigning to genes/species for NC PhyloP, we will need to alter code to do it for just species for ML

gobp = pd.read_csv("GOBP_AccelEvol_Input_FiltForAccelEvol.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("HPO_AccelEvol_Input_FiltForAccelEvol.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")

np.random.seed(10)
x = []
for key in d_BP.keys():
    if len(d_BP[key]) > 20 and len(d_BP[key]) < 200:
        x = d_BP[key] + x
x = list(set(x))

def control(ads, adns, seed = 6):

    # choose bins (quantiles work very well)
    bins = np.quantile(
        ads["Size"],
        q=np.linspace(0, 1, 21)  # 20 bins
    )
    
    
    # drop duplicate bin edges just in case
    bins = np.unique(bins)
    
    # assign bins
    ads["bin"] = pd.cut(ads["Size"], bins=bins, include_lowest=True)
    adns["bin"] = pd.cut(adns["Size"], bins=bins, include_lowest=True)
    
    # how many to sample per bin (match vxcu)
    target_counts = ads["bin"].value_counts()
    
    # downsample vxhu
    adns_matched = (
        adns.copy()
        .groupby("bin", group_keys=False)
        .apply(
            lambda x: x.sample(
                n=min(len(x), target_counts.get(x.name, 0)),
                random_state=seed
            )
        )
        .drop(columns="bin")
    ).copy()
    
    target_props = ads["bin"].value_counts(normalize=True)
    N = len(adns)
    
    def sample_bin(x):
        bin_name = x.name
        if bin_name not in target_props:
            return None
    
        n_bin = int(round(target_props[bin_name] * N))
        if n_bin <= 0:
            return None
    
        # sample *with replacement* only if needed
        replace = n_bin > len(x)
        return x.sample(n=n_bin, replace=replace, random_state=seed)
    
    adns_matched = (
        adns.copy()
        .groupby("bin", group_keys=False)
        .apply(sample_bin)
        .drop(columns="bin")
    ).copy()
    return adns_matched.copy()

import warnings

# Suppress all warnings from Pandas
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
outa = []
for i in ["D50", "D100", "CNCC", "SKM", "HP", "CM", "PP", "RPE", "MN"]:
    if i == "D50" or i == "D100":
        v = pd.read_csv("../Prime_DB_Expression_NoBackup/Agoglia_Fraser_2021/STAR/Hybrid_chpr/Agoglia_Fraser_2021_DESeq2_HumChp_Hybrid_CS_" + i + "_chpr.txt", sep = "\t")
        a = pd.read_csv("Brain_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
        
        vu = v[(v["padj_mine"] < 0.05) & (v["log2FoldChange"] > 0.5)]
        vd = v[(v["padj_mine"] < 0.05) & (v["log2FoldChange"] < -0.5)]
    elif i in ["MN", "CM", "HP", "PP", "RPE", "SKM"]:
        v = pd.read_csv("../Myriad_RNA/" + i + "_Filtered.txt", sep = "\t")
        v["Gene"] = v["gene"]
        if i == "CM":
            a = pd.read_csv("Heart_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
        elif i == "MN":
            a = pd.read_csv("Brain_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
        elif i == "HP":
            a = pd.read_csv("Liver_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
        else:
            a = pd.read_csv("AllTissues_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
        vu = v[(v["padj_mine Chpreffed"] < 0.05) & (v["L2FC Chpreffed"] > 0.5)]
        vd = v[(v["padj_mine Chpreffed"] < 0.05) & (v["L2FC Chpreffed"] < -0.5)]
    elif i == "CNCC":
        v = pd.read_csv("../Prime_DB_Expression_NoBackup/Gokhman_Fraser_2021/STAR/Hybrid_chpr/Gokhman_Fraser_2021_DESeq2_HumChp_Hybrid_" + i + "_chpr.txt", sep = "\t")
        a = pd.read_csv("AllTissues_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
        
        vu = v[(v["padj_mine"] < 0.05) & (v["log2FoldChange"] > 0.5)]
        vd = v[(v["padj_mine"] < 0.05) & (v["log2FoldChange"] < -0.5)]
        
        
    a = a[a["Size"] > 100]
    a = a[a["Variance of ASE Dist"] < 0.5]
    au = a[a["gene"].isin(vu["Gene"])]
    ad = a[a["gene"].isin(vd["Gene"])]
    out = []
    outm = []
    outhl = []
    for j in range(100):
        au_matched = control(ad, au, seed = j)
        outm.append(np.median(ad["Variance of ASE Dist"]) - np.median(au_matched["Variance of ASE Dist"]))
        x = np.array(list(ad["Variance of ASE Dist"]))
        y = np.array(list(au_matched["Variance of ASE Dist"]))
        diffs = (x[:, None] - y[None, :]).flatten()
        hl_estimator = np.median(diffs)
        out.append(mwu(ad["Variance of ASE Dist"], au_matched["Variance of ASE Dist"])[1])
        outhl.append(hl_estimator)
    outa.append([i, np.median(out), outm[np.argmin(np.abs(np.array(out) - np.median(out)))], outhl[np.argmin(np.abs(np.array(out) - np.median(out)))], np.argmin(np.abs(np.array(out) - np.median(out)))])
    print([i, np.median(out), outm[np.argmin(np.abs(np.array(out) - np.median(out)))], outhl[np.argmin(np.abs(np.array(out) - np.median(out)))], np.argmin(np.abs(np.array(out) - np.median(out)))])
df = pd.DataFrame(outa)
df.columns = ["Cell type", "Median p-value", "Median difference in medians", "Hodges-Lehmann estimator", "Arg median p-value"]
df["FDR"] = fdrcorrection(df["Median p-value"])[1]
df.to_csv("ASE_Variance_DownVsUp.csv", index = False)
df

In [ ]:
i = "D100"

if i == "D50" or i == "D100":
    v = pd.read_csv("../Prime_DB_Expression_NoBackup/Agoglia_Fraser_2021/STAR/Hybrid_chpr/Agoglia_Fraser_2021_DESeq2_HumChp_Hybrid_CS_" + i + "_chpr.txt", sep = "\t")
    a = pd.read_csv("Brain_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
    
    vu = v[(v["padj_mine"] < 0.05) & (v["log2FoldChange"] > 0.5)]
    vd = v[(v["padj_mine"] < 0.05) & (v["log2FoldChange"] < -0.5)]
elif i in ["MN", "CM", "HP", "PP", "RPE", "SKM"]:
    v = pd.read_csv("../Myriad_RNA/" + i + "_Filtered.txt", sep = "\t")
    v["Gene"] = v["gene"]
    if i == "CM":
        a = pd.read_csv("Heart_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
    elif i == "MN":
        a = pd.read_csv("Brain_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
    elif i == "HP":
        a = pd.read_csv("Liver_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
    else:
        a = pd.read_csv("AllTissues_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
    vu = v[(v["padj_mine Chpreffed"] < 0.05) & (v["L2FC Chpreffed"] > 0.5)]
    vd = v[(v["padj_mine Chpreffed"] < 0.05) & (v["L2FC Chpreffed"] < -0.5)]
elif i == "CNCC":
    v = pd.read_csv("../Prime_DB_Expression_NoBackup/Gokhman_Fraser_2021/STAR/Hybrid_chpr/Gokhman_Fraser_2021_DESeq2_HumChp_Hybrid_" + i + "_chpr.txt", sep = "\t")
    a = pd.read_csv("AllTissues_MinReads_Per_Allele10Variance_Mean.txt", sep = "\t").dropna()
    
    vu = v[(v["padj_mine"] < 0.05) & (v["log2FoldChange"] > 0.5)]
    vd = v[(v["padj_mine"] < 0.05) & (v["log2FoldChange"] < -0.5)]
j = 12

a = a[a["Size"] > 100]
a = a[a["Variance of ASE Dist"] < 0.5]
au = a[a["gene"].isin(vu["Gene"])]
ad = a[a["gene"].isin(vd["Gene"])]

au_matched = control(ad, au, seed = j)

sns.kdeplot({"Lower expr. from human allele":list(ad["Variance of ASE Dist"]), "Higher expr. from human allele":list(au_matched["Variance of ASE Dist"])}, common_norm = False, fill = True, palette = {"Lower expr. from human allele":"#39BEB1", "Higher expr. from human allele":"#CD99D8"})
#plt.legend(bbox_to_anchor = (1, 1))

In [ ]:
l = pd.DataFrame([np.repeat("Lower expr. from human allele", len(list(ad["Variance of ASE Dist"]))), list(ad["Variance of ASE Dist"])]).T
u = pd.DataFrame([np.repeat("Higher expr. from human allele", len(list(au_matched["Variance of ASE Dist"]))), list(au_matched["Variance of ASE Dist"])]).T

dfp = pd.concat([l, u])
dfp.columns = ["Expression", "ASE variance"]
fig, ax = plt.subplots(figsize = (6, 4), dpi = 600)
#sns.kdeplot(dfp, x = "ASE variance", hue = "Expression", common_norm = False, fill = True, palette = {"Lower expr. from human allele":"#39BEB1", "Higher expr. from human allele":"#CD99D8"})
sns.kdeplot({"Lower expr. from human allele":list(ad["Variance of ASE Dist"]), "Higher expr. from human allele":list(au_matched["Variance of ASE Dist"])}, common_norm = False, fill = True, palette = {"Lower expr. from human allele":"#39BEB1", "Higher expr. from human allele":"#CD99D8"})

#plt.legend(bbox_to_anchor = (1, 1))
plt.ylabel("Density", size = 12)
plt.xlabel("ASE variance", size = 12)
plt.title("Cortical organoids", size = 14)
plt.xticks(size = 11)
plt.yticks(size = 11)

In [ ]:
df = pd.read_csv("ASE_Variance_DownVsUp.csv")
df["-Log$_{10}$(FDR)"] = -np.log10(df["FDR"])

df["Significance"] = ["FDR < 0.05", "FDR < 0.05"] + list(np.repeat("Not significant", 7))
fig, ax = plt.subplots(figsize = (6, 4), dpi = 600)
sns.scatterplot(data = df, x = "Hodges-Lehmann estimator", y = "-Log$_{10}$(FDR)", hue = "Significance", palette = {"FDR < 0.05":"#39BEB1", "Not significant":"grey"})

plt.ylabel("-Log$_{10}$(FDR)", size = 12)
plt.xlabel("Hodges-Lehmann estimator", size = 12)
plt.title("Comparison of ASE variance for genes with\nlower or higher expr. from human allele", size = 14)
plt.xticks(size = 11)
plt.yticks(size = 11)

In [ ]:
df = pd.read_csv("ASE_Variance_DownVsUp.csv")
df